# Assignment C1M1 — Introduction to RAG Systems
---

In this assignment you will build your **first RAG (Retrieval-Augmented Generation) pipeline** using a 2024 news dataset.  
The goal is to let an LLM answer questions about events it was never trained on by retrieving relevant articles first.

**What you will build:**
1. `get_relevant_data` — combine retrieval + lookup in one step
2. `format_relevant_data` — turn raw documents into a clean string for the prompt
3. Watch the full pipeline (`retrieve → format → prompt → LLM`) come together
4. Compare answers **with RAG** vs **without RAG** side-by-side

---
### ⚙️ Prerequisites — check your `.env`

Make sure your `.env` (in the same folder as this notebook) contains at least:

```
LLM_BACKEND=ollama          # or gemini / together
OLLAMA_MODEL=qwen2.5:7b     # only needed for ollama
# GEMINI_API_KEY=...        # only needed for gemini
# TOGETHER_API_KEY=...      # only needed for together
```

---
### 📂 Expected folder layout

```
assignments/
├── C1M1_Assignment.ipynb   ← you are here
├── utils.py
├── unittests.py
├── embeddings.joblib
├── news_data_dedup.csv
└── .env
```

## Table of Contents
- [1 — Introduction](#1)
  - [1.1 RAG architecture overview](#1-1)
  - [1.2 Imports](#1-2)
- [2 — Loading the dataset](#2)
- [3 — Main Functions](#3)
  - [3.1 query_news — lookup by index](#3-1)
  - [3.2 retrieve — semantic search](#3-2)
  - [3.3 Exercise 1 — get_relevant_data](#3-3)
  - [3.4 Exercise 2 — format_relevant_data](#3-4)
  - [3.5 generate_final_prompt — assemble the prompt](#3-5)
  - [3.6 llm_call — full pipeline](#3-6)
- [4 — Experiment with your RAG system](#4)

<a id='1'></a>
## 1 — Introduction

<a id='1-1'></a>
### 1.1 RAG architecture overview

A simplified RAG flow looks like this:

```
User query
    │
    ▼
[ Retriever ]  ──→  finds top-k relevant documents from the knowledge base
    │
    ▼
[ Formatter ]  ──→  turns raw docs into a clean text block
    │
    ▼
[ Prompt builder ] ──→  query + formatted docs → augmented prompt
    │
    ▼
[ LLM ]  ──→  generates a grounded answer
```

You will implement the **Retriever** and **Formatter** steps in the exercises below.

<a id='1-2'></a>
### 1.2 Imports

Run this cell first. It loads your backend-aware `utils.py` and the local test suite.

In [1]:
from utils import (
    retrieve,
    pprint,
    generate_with_single_input,
    read_dataframe,
    display_widget,
)
import unittests

print("✅ Imports successful!")

Loading embedding model... (first run may take a minute)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model and embeddings loaded.
✅ Imports successful!


<a id='2'></a>
## 2 — Loading the dataset

The dataset is [News Headlines 2024](https://www.kaggle.com/datasets/dylanjcastillo/news-headlines-2024) — thousands of headlines from BBC, WSJ, Reuters, and others.

Each record has:

| Field | Description |
|---|---|
| `guid` | Unique article ID |
| `title` | Headline |
| `description` | Short summary |
| `venue` | Publisher |
| `url` | Link to full article |
| `published_at` | Publication date (YYYY-MM-DD) |
| `updated_at` | Last update date |

In [2]:
NEWS_DATA = read_dataframe("news_data_dedup.csv")
print(f"Loaded {len(NEWS_DATA):,} news articles.")

Loaded 870 news articles.


In [3]:
# Inspect a couple of records
pprint(NEWS_DATA[9:11])

[
  {
    "guid": "5dae28f191cfd1047f67c409e616fc3f",
    "title": "Paris's Moulin Rouge loses windmill sails overnight",
    "description": "The cause of the sails' collapse from the roof of the world famous cabaret club is not yet clear.",
    "venue": "BBC",
    "url": "https://www.bbc.co.uk/news/world-europe-68895836",
    "published_at": "2024-04-25",
    "updated_at": "2024-04-26"
  },
  {
    "guid": "d2c3ff79d4e068911d05416ca061cd51",
    "title": "Ukraine uses longer-range US missiles for first time",
    "description": "Missiles secretly delivered this month have been used to strike Russian targets in Crimea, US media say.",
    "venue": "BBC",
    "url": "https://www.bbc.co.uk/news/world-europe-68893196",
    "published_at": "2024-04-25",
    "updated_at": "2024-04-26"
  }
]


> **Notice:** `title`, `description`, `url`, and `published_at` carry the most useful information for answering queries.

<a id='3'></a>
## 3 — Main Functions

Here is the map of what you will build vs what is already provided:

| Function | Status | Purpose |
|---|---|---|
| `query_news(indices)` | ✅ Provided | Look up documents by index |
| `retrieve(query, top_k)` | ✅ Provided (in utils.py) | Semantic search → indices |
| **`get_relevant_data`** | 🔨 **Exercise 1** | Combine retrieve + query_news |
| **`format_relevant_data`** | 🔨 **Exercise 2** | Format docs → string |
| `generate_final_prompt` | ✅ Provided | Build the augmented prompt |
| `llm_call` | ✅ Provided | End-to-end RAG call |

<a id='3-1'></a>
### 3.1 `query_news` — lookup by index

This simple helper takes a list of integer indices and returns the matching documents from `NEWS_DATA`. It is given to you.

In [4]:
def query_news(indices):
    """
    Return the NEWS_DATA records at the given indices.

    Parameters:
        indices (list[int]): positions in NEWS_DATA to retrieve

    Returns:
        list[dict]: the corresponding news records
    """
    return [NEWS_DATA[i] for i in indices]

In [5]:
# Quick check — fetch records at positions 3, 6, 9
pprint(query_news([3, 6, 9]))

[
  {
    "guid": "e696224ac208878a5cec8bdc9f97c632",
    "title": "Europe risks dying and faces big decisions - Macron",
    "description": "The French president delivers a stark warning for Europe to act fast to survive in a changing world.",
    "venue": "BBC",
    "url": "https://www.bbc.co.uk/news/world-europe-68898887",
    "published_at": "2024-04-25",
    "updated_at": "2024-04-26"
  },
  {
    "guid": "4f585bad8f61b715fbafe2f022ab0ae8",
    "title": "Supreme Court divided on whether Trump has immunity",
    "description": "The justices discussed immunity, coups, pardons, Operation Mongoose - and the future of democracy.",
    "venue": "BBC",
    "url": "https://www.bbc.co.uk/news/world-us-canada-68901817",
    "published_at": "2024-04-25",
    "updated_at": "2024-04-26"
  },
  {
    "guid": "5dae28f191cfd1047f67c409e616fc3f",
    "title": "Paris's Moulin Rouge loses windmill sails overnight",
    "description": "The cause of the sails' collapse from the roof of the world famou

<a id='3-2'></a>
### 3.2 `retrieve` — semantic search

The `retrieve` function (from `utils.py`) embeds your query and returns the indices of the most similar documents using cosine similarity.

**Signature:**
```python
retrieve(query: str, top_k: int) -> list[int]
```

You don't need to implement it — just understand what it returns: **a list of integer indices**.

In [6]:
# Try it out
indices = retrieve("Concerts in North America", top_k=1)
print("Indices returned by retrieve:", indices)

Indices returned by retrieve: [np.int64(350)]


In [7]:
# Now look up the actual documents at those indices
retrieved_docs = query_news(indices)
pprint(retrieved_docs)

[
  {
    "guid": "927257674585bb6ef669cf2c2f409fa7",
    "title": "\u2018The working class can\u2019t afford it\u2019: the shocking truth about the money bands make on tour",
    "description": "As Taylor Swift tops $1bn in tour revenue, musicians playing smaller venues are facing pitiful fees and frequent losses. Should the state step in to save our live music scene?When you see a band playing to thousands of fans in a sun-drenched festival field, signing a record deal with a major label or playing endlessly from the airwaves, it\u2019s easy to conjure an image of success that comes with some serious cash to boot \u2013 particularly when Taylor Swift has broken $1bn in revenue for her current Eras tour. But looks can be deceiving. \u201cI don\u2019t blame the public for seeing a band playing to 2,000 people and thinking they\u2019re minted,\u201d says artist manager Dan Potts. \u201cBut the reality is quite different.\u201dPost-Covid there has been significant focus on grassroots mus

<a id='3-3'></a>
## ✏️ Exercise 1 — `get_relevant_data`

**Goal:** wrap `retrieve` + `query_news` into a single convenience function.

**Steps:**
1. Call `retrieve(query, top_k)` → get a list of indices
2. Call `query_news(indices)` → get the matching documents
3. Return the documents

<details>
<summary>💡 Hint 1 — retrieve</summary>

`retrieve(query, top_k=5)` returns a **list of integers** (the indices of the best-matching documents).
</details>

<details>
<summary>💡 Hint 2 — query_news</summary>

`query_news(indices)` takes that list of integers and returns a **list of dicts** (the actual news records).
</details>

In [ ]:
def get_relevant_data(query: str, top_k: int = 5) -> list:
    """
    Retrieve the top_k most relevant news articles for the given query.

    Parameters:
        query  (str) : search query
        top_k  (int) : number of articles to return (default 5)

    Returns:
        list[dict] : the top_k most relevant news records
    """
    ### START CODE HERE ###

    # Step 1: get the indices of the top_k most relevant documents
    relevant_indices = None   # ← replace None with your code

    # Step 2: look up the actual documents using those indices
    relevant_data = None      # ← replace None with your code

    ### END CODE HERE ###

    return relevant_data

In [ ]:
# Manual test — check the output looks sensible
query = "Greatest storms in the US"
result = get_relevant_data(query, top_k=1)
pprint(result)

**Expected output** (your top-1 result should be this article):
```json
[
  {
    "guid": "3ca548fe82c3fcae2c4c0c635d03eb2e",
    "title": "Large tornado seen touching down in Nebraska",
    "description": "Severe and powerful storms have moved across several US states...",
    "venue": "BBC",
    "url": "https://www.bbc.co.uk/news/world-us-canada-68860070",
    "published_at": "2024-04-26",
    "updated_at": "2024-04-28"
  }
]
```

In [ ]:
# Run the automated tests
unittests.test_get_relevant_data(get_relevant_data)

<a id='3-4'></a>
## ✏️ Exercise 2 — `format_relevant_data`

**Goal:** convert a list of news record dicts into a single, readable string that you can paste into a prompt.

**Requirements** (the autograder checks these):
- The output must be a **string**
- It must contain the exact words: `title`, `url`, `published_at`, `description` (case-insensitive)
- Each keyword must appear **once per document** (so 4 docs → 4 occurrences of each keyword)

**Example of an acceptable format:**
```
Title: Large tornado seen touching down in Nebraska
Description: Severe and powerful storms ...
Published_at: 2024-04-26
URL: https://www.bbc.co.uk/...

Title: Another article ...
...
```

<details>
<summary>💡 Hint 1 — accessing fields</summary>

Each `document` in `relevant_data` is a dict. Access fields like:
```python
document['title']
document['description']
document['published_at']
document['url']
```
</details>

<details>
<summary>💡 Hint 2 — building the list</summary>

Start with `formatted_documents = []`, build a formatted string per document inside the loop, then `.append()` it. At the end, return `"\n".join(formatted_documents)`.
</details>

In [ ]:
def format_relevant_data(relevant_data: list) -> str:
    """
    Format a list of news records into a structured string for use in a prompt.

    Parameters:
        relevant_data (list[dict]) : news records from get_relevant_data

    Returns:
        str : a formatted string containing all documents
    """
    ### START CODE HERE ###

    # Create an empty list to collect each formatted document string
    formatted_documents = []  # ← initialise the list (this line is correct already)

    for document in relevant_data:
        # Build a formatted string for this one document.
        # It MUST include: title, description, published_at, url
        formatted_document = None   # ← replace None with your f-string

        # Add the formatted string to the list
        formatted_documents.append(formatted_document)  # ← fill in formatted_document

    ### END CODE HERE ###

    return "\n".join(formatted_documents)

In [ ]:
# Manual test — format 4 documents and print the result
example_data = NEWS_DATA[4:8]
print(format_relevant_data(example_data))

In [ ]:
# Run the automated tests
unittests.test_format_relevant_data(format_relevant_data)

<a id='3-5'></a>
### 3.5 `generate_final_prompt` — assemble the prompt

This function is given to you. It combines your `get_relevant_data` and `format_relevant_data` functions into a complete augmented prompt. Feel free to edit the default prompt template and observe how it changes the LLM's output.

In [ ]:
def generate_final_prompt(query, top_k=5, use_rag=True, prompt=None):
    """
    Build the final prompt to send to the LLM.

    Parameters:
        query   (str)  : user's question
        top_k   (int)  : number of documents to retrieve
        use_rag (bool) : if False, return the bare query (no context injected)
        prompt  (str)  : optional custom template; must contain {query} and {documents}

    Returns:
        str : the prompt ready to send to the LLM
    """
    if not use_rag:
        return query

    relevant_data          = get_relevant_data(query, top_k=top_k)
    retrieve_data_formatted = format_relevant_data(relevant_data)

    if prompt is None:
        prompt = (
            f"Answer the user query below. "
            f"Additional 2024 news context is provided — use it alongside your existing knowledge.\n\n"
            f"Query: {query}\n"
            f"2024 News Context:\n{retrieve_data_formatted}"
        )
    else:
        prompt = prompt.format(query=query, documents=retrieve_data_formatted)

    return prompt

In [ ]:
# See what the augmented prompt looks like
print(generate_final_prompt("Tell me about the US GDP in the past 3 years."))

<a id='3-6'></a>
### 3.6 `llm_call` — the full pipeline

This ties everything together: build the prompt → call the LLM → return the answer.

In [ ]:
def llm_call(query, top_k=5, use_rag=True, prompt=None):
    """
    Full RAG pipeline: retrieve → format → prompt → LLM.

    Parameters:
        query   (str)  : question to answer
        top_k   (int)  : documents to retrieve
        use_rag (bool) : True = RAG-augmented, False = bare LLM
        prompt  (str)  : optional custom prompt template

    Returns:
        str : the LLM's answer
    """
    final_prompt      = generate_final_prompt(query, top_k, use_rag, prompt)
    generated_response = generate_with_single_input(final_prompt)
    return generated_response["content"]

In [ ]:
query = "Tell me about the US GDP in the past 3 years."

print("=== WITH RAG ===")
print(llm_call(query, use_rag=True))

In [ ]:
print("=== WITHOUT RAG ===")
print(llm_call(query, use_rag=False))

<a id='4'></a>
## 4 — Experiment with your RAG System

The widget below lets you try any query and compare RAG vs non-RAG answers side by side.

**Suggested queries to try:**
- *What were the most important events of 2024?*
- *How is global warming progressing in 2024?*
- *Tell me about the most recent advances in AI.*
- *Give me the most important facts from the past year.*

You can also provide a **custom prompt layout** using `{query}` and `{documents}` as placeholders:
```
Answer this question: {query}
Use only the following context: {documents}
```

In [ ]:
display_widget(llm_call)

---
🎉 **Congratulations!** You have built your first RAG pipeline from scratch.

**What you implemented:**
- `get_relevant_data` — seamlessly combining semantic retrieval with document lookup
- `format_relevant_data` — shaping raw data into LLM-ready context

**What the pipeline does end-to-end:**
```
query → retrieve (embeddings) → query_news → format → augmented prompt → LLM → answer
```

In the next module you will dive deeper into retrieval techniques, embeddings, and vector databases.